| 方法 | 模型收到什麼？ |
|---|---|
| 單次修補 | 題目與有 bug 的程式，只生成一次。 |
| Retry Only | 從第一次答案出發，只提醒「請再檢查一次」。 |
| Raw Feedback | 從第一次答案出發，提供 unittest 原始輸出。 |

在另一個 terminal 執行 `ollama serve`。模型下載好後，這份教材只會呼叫本機服務，不需要 API key。

要先測初始程式，可執行 `python -m unittest discover -s tests -p test_csv_parser.py -v`。初始版本有測試失敗是預期結果。

In [1]:
import json
import tempfile
import time
from difflib import unified_diff
from html import escape
from pathlib import Path

from IPython.display import Code, HTML, Markdown, display
from workshop_tools import (
    DEV_CASES, DEMO_CODE,
    ask_ollama, extract_code, run_tests, write_solution,
)

MODE = "ollama" 
MODEL = "qwen3.5:4b"
THINK = False
OPTIONS = {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "min_p": 0,
           "seed": 7, "num_ctx": 4096, "num_predict": 2048}
MAX_ATTEMPTS = 2  # 包含共同的第一次；兩組各最多再修一次。
STYLES = {"retry": "Retry Only", "raw": "Raw Feedback"}


CSV 用逗號分隔欄位，但引號內的逗號是內容，兩個連續雙引號代表一個雙引號。

- `a,"b,c",d` → `["a", "b,c", "d"]`
- `"a""b",c` → `['a"b', "c"]`
- `a,b,` → `["a", "b", ""]`

空欄位和前後空白都要保留，空輸入回傳 `[""]`。輸入保證合法且只有一行，不需要處理錯誤格式，也不能使用 `csv`／`_csv` library。

In [2]:
TASK = """修好 parse_csv(line: str) -> list[str]，解析一行合法 CSV。
逗號分隔欄位；完整 quoted field 內的逗號屬於內容；quoted field 內兩個連續雙引號表示一個雙引號。
保留所有空欄位，包含開頭、連續、結尾逗號與空 quoted field。空輸入回傳 [""]。
保留欄位前後的空白，不要自動 strip。雙引號只出現在完整 quoted field 或其中的 escaped quote。
不必處理 multiline fields 或格式錯誤。不可使用 Python csv library（也不可使用 _csv）。"""

BUGGY_CODE = Path("csv_parser.py").read_text(encoding="utf-8")
INITIAL_MESSAGES = [
    {"role": "system", "content": "你正在修一個 Python 函式。只輸出完整的 Python 程式碼，包含所需 import，不要解釋，不要改測試。每個函式保留一句簡短中文 docstring。"},
    {"role": "user", "content": TASK + "\n目前程式：\n" + BUGGY_CODE},
]
display(Code(BUGGY_CODE, language="python"))
initial_result = run_tests(BUGGY_CODE, DEV_CASES)
print(f"初始程式：{initial_result['count']}/{initial_result['total']} 通過。")


"""CSV Parser 修復練習：這份起始程式刻意保留 bug。

輸入都是合法的單行 CSV，不包含多行欄位，也不必處理格式錯誤。
逗號分隔欄位，但完整 quoted field 裡的逗號屬於內容。
quoted field 內的兩個連續雙引號代表一個雙引號。
請保留空欄位與前後空白；空輸入也應回傳一個空欄位。
修復時不能直接使用 Python csv library。
"""


def parse_csv(line: str) -> list[str]:
    """逐字讀取欄位；這是刻意保留邊界錯誤的起始版本。"""
    result = []
    field = ""
    in_quotes = False

    for ch in line:
        if ch == '"':
            in_quotes = not in_quotes
        elif ch == ',' and not in_quotes:
            result.append(field)
            field = ""
        else:
            field += ch

    if field:
        result.append(field)

    return result

初始程式：6/10 通過。


模型只產生文字；`run_tests()` 才會把文字寫進暫存 `solution.py`，用新程序執行 unittest。迴圈依測試結果判斷是否停止；Raw Feedback 還會把原始輸出送回模型。

暫存目錄與寫檔限制類似於 sandbox 的概念，避免模型改壞。

In [3]:
def ask_model(messages):
    if MODE == "demo":
        has_answer = any(message["role"] == "assistant" for message in messages)
        return DEMO_CODE if has_answer else BUGGY_CODE
    if MODE != "ollama":
        raise ValueError('MODE 只能是 "ollama" 或 "demo"。')
    return ask_ollama(messages, MODEL, OPTIONS, think=THINK)


模型先修一次，測試只評分、不回傳。下方的 `first` 會作為兩組共同起點；後面重跑實驗時，可以直接沿用它。

In [4]:
print("正在生成共同的第一次答案……", flush=True)
started = time.perf_counter()
first_reply = ask_model(INITIAL_MESSAGES)
first_code = extract_code(first_reply)
first = {"reply": first_reply, "code": first_code,
         "result": run_tests(first_code, DEV_CASES),
         "seconds": time.perf_counter() - started}
display(Code(first["code"], language="python"))
print(f"第一次修補：{first['result']['count']}/{first['result']['total']} 通過。")


正在生成共同的第一次答案……


"""CSV 解析器：逐字讀取欄位，處理引號與空白。"""


def parse_csv(line: str) -> list[str]:
    """逐字讀取欄位；這是刻意保留邊界錯誤的起始版本。"""
    result = []
    field = ""
    in_quotes = False

    for ch in line:
        if ch == '"':
            if in_quotes and field.endswith('"'):
                # 遇到連續雙引號，視為轉義字符，不切換狀態
                field += ch
            else:
                in_quotes = not in_quotes
        elif ch == ',' and not in_quotes:
            result.append(field)
            field = ""
        else:
            field += ch

    if field:
        result.append(field)

    return result

第一次修補：6/10 通過。


## 兩種不同策略

Retry Only 沒有具體失敗資訊；Raw 原樣回傳 unittest 輸出，讓模型看見哪些案例失敗。

兩組用相同測試評分，差別在於下一次生成時，模型是否收到測試結果。

In [5]:
def format_feedback(result, style):
    """重試時可以只給提醒，也可以把測試輸出交給模型。"""
    if style == "retry":
        return "請再檢查一次，修正可能的錯誤。"
    if style == "raw":
        return result["raw"]
    raise ValueError("未知的回饋方式。")


In [6]:
for style, label in STYLES.items():
    feedback = format_feedback(first["result"], style)
    display(HTML(f"<details><summary>{label}</summary>"
                 f"<pre style='white-space:pre-wrap'>{escape(feedback)}</pre></details>"))


兩組共用下面的 `repair()`。每次呼叫都從 `first` 建立新對話，不會把其他組的修正版接過來；通過開發測試就停，否則最多生成 3 次。

In [7]:
def repair(first, feedback_style="raw", max_attempts=2):
    """從共同答案開始，只替換回饋格式，通過或用完次數就停止。"""
    messages = [message.copy() for message in INITIAL_MESSAGES]
    messages.append({"role": "assistant", "content": first["reply"]})
    history = [first]
    for attempt in range(2, max_attempts + 1):
        if history[-1]["result"]["passed"]:
            break
        feedback = format_feedback(history[-1]["result"], feedback_style)
        prompt = TASK + "\n\n" + feedback + "\n請修正程式，只回傳完整 Python 程式碼。"
        messages.append({"role": "user", "content": prompt})
        print(f"{STYLES[feedback_style]}：第 {attempt} 次生成……", flush=True)
        started = time.perf_counter()
        reply = ask_model(messages)
        code = extract_code(reply)
        result = run_tests(code, DEV_CASES)
        history.append({"reply": reply, "code": code, "result": result,
                        "seconds": time.perf_counter() - started, "prompt": prompt})
        messages.append({"role": "assistant", "content": reply})
        print(f"{result['count']}/{result['total']} 通過。", flush=True)
    return history


In [8]:
# 每組都拿同一個 first，不能接著上一組的最後答案繼續修。
experiments = {}
for style in STYLES:
    experiments[style] = repair(first, feedback_style=style, max_attempts=MAX_ATTEMPTS)


Retry Only：第 2 次生成……
7/10 通過。
Raw Feedback：第 2 次生成……
7/10 通過。


## 比較成績

In [9]:
summary = []
groups = [("單次修補（共同起點）", [first])] + [(STYLES[style], records) for style, records in experiments.items()]
for label, records in groups:
    summary.append({"group": label, "attempts": len(records),
                    "seconds": sum(record["seconds"] for record in records),
                    "development": records[-1]["result"]})

table = "| 方法 | 生成次數 | 通過題數 | 累計秒數 |\n|---|---:|---:|---:|\n"
for item in summary:
    dev = item["development"]
    table += f"| {item['group']} | {item['attempts']} | {dev['count']}/{dev['total']} | {item['seconds']:.1f} |\n"
display(Markdown(table))
print("模式：", MODE, "；模型：", MODEL)
print("合計生成次數（共同起點只算一次）：", 1 + sum(len(records) - 1 for records in experiments.values()))
if MODE == "demo":
    print("以上是手寫流程示例，不能據此判斷哪一種回饋比較有效。")

# 每組保存實際送回的 prompt 與程式，方便核對。
Path("runs").mkdir(exist_ok=True)
trace_path = Path("runs") / f"csv-feedback-{MODE}-{time.time_ns()}.json"
trace = {"benchmark": "csv-feedback", "mode": MODE, "model": MODEL, "options": OPTIONS,
         "think": THINK, "task": TASK, "original": BUGGY_CODE, "initial_messages": INITIAL_MESSAGES,
         "max_attempts": MAX_ATTEMPTS, "development_cases": DEV_CASES,
         "first": first, "experiments": experiments, "summary": summary}
trace_path.write_text(json.dumps(trace, ensure_ascii=False, indent=2), encoding="utf-8")
print("紀錄已保存：", trace_path)


| 方法 | 生成次數 | 通過題數 | 累計秒數 |
|---|---:|---:|---:|
| 單次修補（共同起點） | 1 | 6/10 | 66.8 |
| Retry Only | 2 | 7/10 | 140.9 |
| Raw Feedback | 2 | 7/10 | 180.3 |


模式： ollama ；模型： qwen3.5:4b
合計生成次數（共同起點只算一次）： 3
紀錄已保存： runs/csv-feedback-ollama-1793154041235055786.json


In [10]:
for style, records in experiments.items():
    diff = "".join(unified_diff(first["code"].splitlines(True), records[-1]["code"].splitlines(True),
                                fromfile="共同起點", tofile=STYLES[style])) or "程式沒有改動。"
    display(HTML(f"<details><summary>{STYLES[style]}：程式差異</summary>"
                 f"<pre style='white-space:pre-wrap'>{escape(diff)}</pre></details>"))


## Harness 潛在問題
下面直接用有 bug 的初始程式。只測一般欄位和引號內逗號時，它看起來都對；再加入結尾空欄與 escaped quote，錯誤就出現了。

In [12]:
print("只測基本案例：")
print(run_tests(BUGGY_CODE, DEV_CASES[:2])["feedback"])
print("\n加上結尾空欄與 escaped quote：")
print(run_tests(BUGGY_CODE, [DEV_CASES[4], DEV_CASES[6]])["feedback"])


只測基本案例：
2/2 個測試通過。


加上結尾空欄與 escaped quote：
0/2 個測試通過。
test_01
Input: 'a,b,'
Expected: ['a', 'b', '']
Actual: ['a', 'b']
test_02
Input: '"a""b",c'
Expected: ['a"b', 'c']
Actual: ['ab', 'c']


In [13]:
with tempfile.TemporaryDirectory(prefix="permission-demo-") as folder:
    test_path = Path(folder) / "test_solution.py"
    original_test = "assert parse_csv('a,b') == ['a', 'b']\n"
    test_path.write_text(original_test, encoding="utf-8")
    try:
        write_solution(folder, "test_solution.py", "# 把測試刪掉就不會失敗了\n")
    except PermissionError as error:
        print("修改被拒絕：", error)
    print("測試內容仍然完整：", test_path.read_text(encoding="utf-8") == original_test)


修改被拒絕： 只能修改 solution.py，測試與其他檔案不開放。
測試內容仍然完整： True


## 版本控制：總分不下降才接受

模型修改後的版本不一定比較好，因此加一個直觀的決策：**同一組開發測試中，通過題數增加或持平就接受，減少就保留舊版。**

例如舊版是 6/10，新版 7/10 或 6/10 都接受；新版只有 5/10，就保留原來的 6/10。

下面沿用前面已產生、已跑完全部開發測試的答案，只比較採用決策，不再呼叫模型。這樣可以直接看出：同一份候選答案，加上阻擋規則後，最後留下什麼。


In [14]:
def keep_best_score(current, candidate):
    """比較同一組開發測試；分數增加或持平就接受，下降才保留舊版。"""
    if candidate["result"]["count"] >= current["result"]["count"]:
        return candidate
    return current


In [15]:
# 用每組最後的候選與共同起點比較，採用結果另外存到 protected。
protected = {}
guard_table = "| 方法 | 原版 | 候選 | 決定 | 保留版本 |\n|---|---:|---:|---|---:|\n"
for style, records in experiments.items():
    candidate = records[-1]
    selected = keep_best_score(first, candidate)
    protected[style] = selected
    decision = "採用候選" if selected is candidate else "分數下降，保留原版"
    if len(records) == 1:
        decision = "原版已通過，沒有重試"
    total = first["result"]["total"]
    guard_table += (f"| {STYLES[style]} | {first['result']['count']}/{total} | "
                    f"{candidate['result']['count']}/{total} | {decision} | "
                    f"{selected['result']['count']}/{total} |\n")
display(Markdown(guard_table))

| 方法 | 原版 | 候選 | 決定 | 保留版本 |
|---|---:|---:|---|---:|
| Retry Only | 6/10 | 7/10 | 採用候選 | 7/10 |
| Raw Feedback | 6/10 | 7/10 | 採用候選 | 7/10 |
